# k08 — Pre-registered robustness analyses (STAGE2_DESIGN_FROZEN_v1.0 §6 D4, §9 sensitivity)
**Cannot alter RESULTS_FROZEN_v1.0.** Frozen k03 selections (configuration, width, epochs / boosting rounds), 5 seeds, no retuning, no threshold selection, no target-domain calibration.
- **D4 ablation** (mandatory): payload-length-derived features removed for every model (frame payload length; node mean/max payload length and payload-length-change indicator), W = 64, all cells.
- **Window sensitivity**: W = 32 and W = 128 on the B cells; strides keep the frozen ratio (train W/2, evaluation W, non-overlapping).
- **Negatives-in-attack-span sensitivity**: evaluation-only on the frozen k05 scores; negative windows between the first and last positive window of each recording are removed.
Statistics: per-cell and per-attack-token PR-AUC; exact token-stratified recording bootstrap on the B cells (k06 code unchanged); change versus the frozen values. Operating points are not computed (thresholds would require new out-of-fold runs).

In [ ]:
import os, glob
os.makedirs('/kaggle/working/code', exist_ok=True); os.makedirs('/kaggle/temp', exist_ok=True)
FILES = {'featsW.py': 'import numpy as np, pandas as pd, re, os, time\nW = 64\n\ndef set_W(w):\n    """Window length (frames). Default 64 = frozen design; 32/128 only for the pre-registered window-sensitivity analysis."""\n    global W\n    W = int(w)\nHEXV = np.full(256, 0, dtype=np.uint8)\nfor i, ch in enumerate(\'0123456789abcdef\'):\n    HEXV[ord(ch)] = i; HEXV[ord(ch.upper())] = i\nPOP = np.array([bin(i).count(\'1\') for i in range(256)], dtype=np.uint8)\n\ndef slog(x):\n    return np.sign(x) * np.log1p(np.abs(x))\n\ndef load_file(path):\n    df = pd.read_csv(path, dtype={\'arbitration_id\': str, \'data_field\': str, \'attack\': np.int8}, keep_default_na=True)\n    ts = df[\'timestamp\'].to_numpy(np.float64)\n    ids = df[\'arbitration_id\'].str.rjust(3, \'0\').str[-3:]\n    ib = np.frombuffer(\'\'.join(ids.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 3)\n    idv = HEXV[ib].astype(np.int32)\n    id_int = idv[:, 0] * 256 + idv[:, 1] * 16 + idv[:, 2]\n    d = df[\'data_field\'].fillna(\'\')\n    plen = (d.str.len().to_numpy() // 2).clip(0, 8).astype(np.int8)\n    d16 = d.str[:16].str.ljust(16, \'0\')\n    db = np.frombuffer(\'\'.join(d16.tolist()).encode(\'ascii\'), dtype=np.uint8).reshape(-1, 16)\n    nib = HEXV[db]\n    pay = (nib[:, 0::2] * 16 + nib[:, 1::2]).astype(np.uint8)\n    posmask = np.arange(8)[None, :] < plen[:, None]\n    pay = np.where(posmask, pay, 0).astype(np.uint8)\n    y = df[\'attack\'].to_numpy(np.int8)\n    return ts, id_int, plen, pay, y\n\ndef per_frame_globals(ts, id_int, plen, pay):\n    n = len(ts); idx = np.arange(n)\n    order = np.lexsort((idx, id_int))\n    prev = np.full(n, -1, dtype=np.int64)\n    same = np.r_[False, id_int[order][1:] == id_int[order][:-1]]\n    prev[order[same]] = order[np.flatnonzero(same) - 1]\n    has = prev >= 0\n    pp = np.where(has, prev, 0)\n    dt_same = np.where(has, ts - ts[pp], 0.0)\n    x = pay ^ pay[pp]\n    ham = np.where(has, POP[x].sum(1), 0).astype(np.float32)\n    maxlen = np.maximum(plen, plen[pp]).astype(np.float32)\n    chg = np.where(has, (x != 0).sum(1) / np.maximum(maxlen, 1), 0).astype(np.float32)\n    lenchg = np.where(has, plen != plen[pp], False)\n    ent = np.zeros(n, dtype=np.float32)\n    for s in range(0, n, 500000):\n        b = pay[s:s + 500000]; L = plen[s:s + 500000].astype(np.int32)\n        valid = np.arange(8)[None, :] < L[:, None]\n        eq = (b[:, :, None] == b[:, None, :]) & valid[:, :, None] & valid[:, None, :]\n        c = eq.sum(2).astype(np.float32)\n        Lf = np.maximum(L, 1).astype(np.float32)[:, None]\n        with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n            term = np.where(valid, np.log2(np.where(c > 0, c, 1) / Lf), 0.0)\n        ent[s:s + 500000] = -(term.sum(1) / Lf[:, 0])\n    return prev, dt_same, ham, chg, ent, lenchg\n\nFRAME_NAMES = [\'plen\', \'dt_prev_any\', \'dt_same\', \'no_prev_same_in_window\', \'hamming\', \'changed_frac\', \'entropy\', \'same_as_prev\']\nNODE_NAMES = [\'count\', \'first_pos\', \'last_pos\', \'ia_mean\', \'ia_min\', \'ia_max\', \'ia_missing\', \'plen_mean\', \'plen_max\', \'plen_changes\', \'ham_mean\', \'chg_mean\', \'ent_mean\']\nGLOBAL_NAMES = [\'duration\', \'distinct_ids\', \'distinct_transitions\', \'fps\']\n\ndef windows_for_file(path, stride, id_perm=None):\n    ts, id_int, plen, pay, y = load_file(path)\n    if id_perm is not None:\n        id_int = id_perm[id_int]\n    n = len(ts)\n    if n < W:\n        return None\n    prev, dt_same_g, ham_g, chg_g, ent_g, lenchg_g = per_frame_globals(ts, id_int, plen, pay)\n    starts = np.arange(0, n - W + 1, stride)\n    nw = len(starts)\n    I = starts[:, None] + np.arange(W)[None, :]\n    ok = prev[I] >= starts[:, None]\n    tsw = ts[I]\n    dtp = np.diff(tsw, axis=1, prepend=tsw[:, :1])\n    idw = id_int[I]\n    fr = np.zeros((nw, W, len(FRAME_NAMES)), dtype=np.float32)\n    fr[..., 0] = plen[I] / 8.0\n    fr[..., 1] = slog(dtp * 1000)\n    fr[..., 2] = np.where(ok, slog(dt_same_g[I] * 1000), 0)\n    fr[..., 3] = ~ok\n    fr[..., 4] = np.where(ok, ham_g[I] / 64.0, 0)\n    fr[..., 5] = np.where(ok, chg_g[I], 0)\n    fr[..., 6] = ent_g[I] / 3.0\n    fr[:, 1:, 7] = idw[:, 1:] == idw[:, :-1]\n    # nodes\n    key = (np.arange(nw)[:, None] * 4096 + idw).ravel()\n    uk, inv = np.unique(key, return_inverse=True)\n    inv = inv.reshape(nw, W)\n    win_of_node = uk // 4096\n    node_first = np.searchsorted(win_of_node, np.arange(nw))\n    local = inv - node_first[:, None]\n    nn = np.bincount(win_of_node, minlength=nw)\n    G = len(uk)\n    fl = inv.ravel()\n    pos = np.broadcast_to(np.arange(W), (nw, W)).ravel()\n    okf = ok.ravel()\n    def agg_sum(v, m=None):\n        return np.bincount(fl, weights=(v if m is None else v * m), minlength=G)\n    order = np.argsort(fl, kind=\'stable\'); fs = fl[order]\n    bnd = np.flatnonzero(np.r_[True, fs[1:] != fs[:-1]])\n    def agg_min(v): return np.minimum.reduceat(v[order], bnd)\n    def agg_max(v): return np.maximum.reduceat(v[order], bnd)\n    cnt = np.bincount(fl, minlength=G).astype(np.float32)\n    iak = np.where(okf, slog(dt_same_g[I].ravel() * 1000), np.nan)\n    nia = agg_sum(okf.astype(np.float64))\n    has_ia = nia > 0\n    ia_mean = np.where(has_ia, agg_sum(np.nan_to_num(iak)) / np.maximum(nia, 1), 0)\n    ia_min = np.where(has_ia, agg_min(np.where(okf, iak, np.inf)), 0)\n    ia_max = np.where(has_ia, agg_max(np.where(okf, iak, -np.inf)), 0)\n    pl = (plen[I].ravel()).astype(np.float64)\n    nd = np.zeros((G, len(NODE_NAMES)), dtype=np.float32)\n    nd[:, 0] = cnt / W\n    nd[:, 1] = agg_min(pos.astype(np.float64)) / W\n    nd[:, 2] = agg_max(pos.astype(np.float64)) / W\n    nd[:, 3] = ia_mean; nd[:, 4] = ia_min; nd[:, 5] = ia_max\n    nd[:, 6] = ~has_ia\n    nd[:, 7] = agg_sum(pl) / cnt / 8.0\n    nd[:, 8] = agg_max(pl) / 8.0\n    nd[:, 9] = agg_max(pl) != agg_min(pl)\n    nd[:, 10] = np.where(has_ia, agg_sum(ham_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1) / 64.0, 0)\n    nd[:, 11] = np.where(has_ia, agg_sum(chg_g[I].ravel().astype(np.float64), okf) / np.maximum(nia, 1), 0)\n    nd[:, 12] = agg_sum(ent_g[I].ravel().astype(np.float64)) / cnt / 3.0\n    node = np.zeros((nw, W, len(NODE_NAMES)), dtype=np.float32)\n    node[win_of_node, np.arange(G) - node_first[win_of_node]] = nd\n    # edges: local src/dst per transition\n    src = local[:, :-1].astype(np.uint8); dst = local[:, 1:].astype(np.uint8)\n    WW = W * W\n    tr = np.unique((np.arange(nw)[:, None] * WW + local[:, :-1] * W + local[:, 1:]).ravel())\n    ntr = np.bincount(tr // WW, minlength=nw)\n    dur = tsw[:, -1] - tsw[:, 0]\n    glob = np.stack([slog(dur * 1000), nn / W, ntr / float(W - 1), slog(W / np.maximum(dur, 1e-6))], 1).astype(np.float32)\n    lab = (y[I].max(1) > 0).astype(np.int8)\n    nattack = y[I].sum(1).astype(np.int16)\n    return dict(frame=fr.astype(np.float16), node=node.astype(np.float16), nmask=(np.arange(W)[None, :] < nn[:, None]),\n                src=src, dst=dst, glob=glob, y=lab, nattack=nattack, starts=starts.astype(np.int64), t0=tsw[:, 0], t1=tsw[:, -1])\n\nSTRUCT_NAMES = [\'transition_entropy\', \'unique_transition_ratio\', \'self_loop_ratio\', \'mean_out_degree\',\n                \'max_out_degree\', \'max_in_degree\', \'degree_entropy\', \'density\']\n\ndef structural_features(src, dst, nmask):\n    """Explicit structural/topological summaries of each window\'s directed transition multigraph.\n    src, dst: (nw, 63) local node indices of consecutive frames; nmask: (nw, 64) valid nodes.\n    Uses only ID-free graph structure (local node indices are arbitrary labels)."""\n    nw, E = src.shape\n    Wn = nmask.shape[1]; WW = Wn * Wn\n    n = nmask.sum(1).astype(np.float64)\n    s = src.astype(np.int64); d = dst.astype(np.int64)\n    w = np.repeat(np.arange(nw), E)\n    key = w * WW + (s * Wn + d).ravel()\n    uk, cnt = np.unique(key, return_counts=True)\n    uw = uk // WW; us = (uk % WW) // Wn; ud = (uk % WW) % Wn\n    p = cnt / float(E)\n    ent = np.bincount(uw, weights=-p * np.log2(p), minlength=nw) / np.log2(E)\n    uniq = np.bincount(uw, minlength=nw) / float(E)\n    selfr = (s == d).sum(1) / float(E)\n    ns = us != ud\n    outdeg = np.bincount(uw[ns] * Wn + us[ns], minlength=nw * Wn).reshape(nw, Wn).astype(np.float64)\n    indeg = np.bincount(uw[ns] * Wn + ud[ns], minlength=nw * Wn).reshape(nw, Wn).astype(np.float64)\n    e_ns = np.bincount(uw[ns], minlength=nw).astype(np.float64)\n    mean_out = np.where(n > 0, e_ns / np.maximum(n, 1), 0)\n    tot = outdeg + indeg; ts = tot.sum(1, keepdims=True)\n    pd_ = np.where(ts > 0, tot / np.maximum(ts, 1), 0)\n    with np.errstate(divide=\'ignore\', invalid=\'ignore\'):\n        h = -(np.where(pd_ > 0, pd_ * np.log2(np.where(pd_ > 0, pd_, 1)), 0)).sum(1)\n    deg_ent = np.where(n > 1, h / np.log2(np.maximum(n, 2)), 0)\n    dens = np.where(n > 1, e_ns / np.maximum(n * (n - 1), 1), 0)\n    return np.stack([ent, uniq, selfr, mean_out, outdeg.max(1), indeg.max(1), deg_ent, dens], 1).astype(np.float32)\n', 'modelsW.py': "import torch, torch.nn as nn, numpy as np, time\nW = 64\n\ndef set_W(w):\n    global W\n    W = int(w)\n\ndef mlp(i, h, o, drop):\n    return nn.Sequential(nn.Linear(i, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, o))\n\nclass Head(nn.Module):\n    def __init__(self, h, g, drop):\n        super().__init__(); self.rho = mlp(3 * h + g, h, 1, drop)\n    def forward(self, H, mask, glob):\n        m = mask.unsqueeze(-1).float()\n        s = (H * m).sum(1); mean = s / m.sum(1).clamp(min=1)\n        mx = H.masked_fill(m == 0, -1e4).max(1).values\n        return self.rho(torch.cat([s, mean, mx, glob], 1)).squeeze(-1)\n\nclass DeepSets(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.phi = nn.Sequential(nn.Linear(f, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, h), nn.ReLU())\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        return self.head(self.phi(b['node']), b['nmask'], b['glob'])\n\nclass SAGELayer(nn.Module):\n    def __init__(self, i, o):\n        super().__init__(); self.self_lin = nn.Linear(i, o); self.nei_lin = nn.Linear(i, o, bias=False)\n    def forward(self, H, A):\n        # A[b, src, dst] = weight; aggregate incoming neighbours of each dst node (weighted mean)\n        agg = torch.bmm(A.transpose(1, 2), H)\n        deg = A.sum(1).unsqueeze(-1)\n        agg = agg / deg.clamp(min=1e-9)\n        return self.self_lin(H) + self.nei_lin(agg)\n\nclass GraphSAGE(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.l1 = SAGELayer(f, h); self.l2 = SAGELayer(h, h); self.drop = nn.Dropout(drop)\n        self.head = Head(h, g, drop)\n    def forward(self, b):\n        A = b['adj']; m = b['nmask'].unsqueeze(-1).float()\n        H = torch.relu(self.l1(b['node'], A)) * m\n        H = torch.relu(self.l2(self.drop(H), A)) * m\n        return self.head(H, b['nmask'], b['glob'])\n\nclass GRUNet(nn.Module):\n    def __init__(self, f, h, g, drop=0.0):\n        super().__init__()\n        self.gru = nn.GRU(f, h, batch_first=True); self.out = mlp(2 * h + g, h, 1, drop)\n    def forward(self, b):\n        O, hn = self.gru(b['frame'])\n        return self.out(torch.cat([hn[-1], O.mean(1), b['glob']], 1)).squeeze(-1)\n\ndef nparams(m):\n    return sum(p.numel() for p in m.parameters())\n\ndef build_adj(src, dst, B, device):\n    A = torch.zeros(B, W, W, device=device)\n    bi = torch.arange(B, device=device).unsqueeze(1).expand_as(src)\n    A.index_put_((bi.reshape(-1), src.reshape(-1).long(), dst.reshape(-1).long()), torch.full((src.numel(),), 1.0 / (W - 1), device=device), accumulate=True)\n    return A\n\ndef rewire_dst(dst, gen):\n    # degree-preserving directed rewiring: permute destination endpoints among a window's 63 edges\n    # (every source keeps its out-degree, every destination keeps its in-degree, multiplicities included)\n    r = torch.rand(dst.shape, generator=gen, device=dst.device)\n    perm = r.argsort(1)\n    return torch.gather(dst, 1, perm)\n\ndef edge_change_fraction(src, dst, dst2):\n    # fraction of the 63 directed edges (as a multiset per window) not present in the original\n    B = src.shape[0]\n    k1 = (src.long() * 64 + dst.long()).sort(1).values\n    k2 = (src.long() * 64 + dst2.long()).sort(1).values\n    fr = []\n    for i in range(B):\n        a, ca = torch.unique(k1[i], return_counts=True); b2, cb = torch.unique(k2[i], return_counts=True)\n        common = 0\n        d = dict(zip(a.tolist(), ca.tolist()))\n        for kk, cc in zip(b2.tolist(), cb.tolist()):\n            common += min(cc, d.get(kk, 0))\n        fr.append(1 - common / 63.0)\n    return float(np.mean(fr))\n", 'exp_robust.py': '"""Stage 7 — pre-registered robustness analyses (STAGE2_DESIGN_FROZEN_v1.0 §6 D4, §9 sensitivity). Frozen k03 selections, 5 seeds,\nNO retuning, NO threshold selection, NO target calibration. Cannot alter RESULTS_FROZEN_v1.0.\n  --mode d4   : W=64; payload-length-derived features (frame plen; node plen_mean, plen_max, plen_changes) removed for ALL models\n                (implemented by setting them to 0 before normalisation = removal: constant inputs carry no information), all cells.\n  --mode w32  : W=32  (train stride 16, eval stride 32), B cells.\n  --mode w128 : W=128 (train stride 64, eval stride 128), B cells.\nUsage: python exp_robust.py --mode d4 --sets set_01,set_03 --device cuda:0"""\nimport os, sys, re, json, time, glob, hashlib, argparse, traceback, zipfile\nimport numpy as np, torch\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport featsW as feats, modelsW as models\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--mode\', required=True, choices=[\'d4\', \'w32\', \'w128\']); ap.add_argument(\'--sets\', required=True)\nap.add_argument(\'--device\', default=\'cuda:0\'); ap.add_argument(\'--out\', default=\'/kaggle/working/robust\')\nap.add_argument(\'--seeds\', default=\'0,1,2,3,4\'); ap.add_argument(\'--lgb_threads\', type=int, default=2)\nargs = ap.parse_args()\nMODE = args.mode; DEV = args.device; SETS = args.sets.split(\',\'); SEEDS = [int(s) for s in args.seeds.split(\',\')]\nW = {\'d4\': 64, \'w32\': 32, \'w128\': 128}[MODE]; TRAIN_STRIDE = W // 2; EVAL_STRIDE = W; BS = 1024\nfeats.set_W(W); models.set_W(W)\nOUT = os.path.join(args.out, MODE); os.makedirs(OUT, exist_ok=True)\nZIP = glob.glob(\'/kaggle/input/**/can-train-and-test-v1.zip\', recursive=True)[0]\nHASHES = glob.glob(\'/kaggle/input/**/file_hashes_sha256.csv\', recursive=True)[0]\nDATA = \'/kaggle/temp/data\'\nEXCLUDED = {(\'set_01\', \'test_04\')}\nMODELS = (\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\')\nD4_FRAME = [feats.FRAME_NAMES.index(\'plen\')]\nD4_NODE = [feats.NODE_NAMES.index(n) for n in (\'plen_mean\', \'plen_max\', \'plen_changes\')]\nCONST_STD = 1.5e-6; EVAL_REWIRE_SEED = 12345\nLOG = open(os.path.join(OUT, f\'log_{"_".join(SETS)}.txt\'), \'a\')\ndef log(*a):\n    s = time.strftime(\'%H:%M:%S \') + f\'[{MODE}] \' + \' \'.join(str(x) for x in a); print(s, flush=True); LOG.write(s + \'\\n\'); LOG.flush()\n\nH = {}\nfor line in open(HASHES).read().strip().split(\'\\n\')[1:]:\n    rel, size, sha = line.split(\',\'); H[rel] = (int(size), sha)\ndef fam(n): return re.sub(r\'-\\d+\\.csv$\', \'\', os.path.basename(n))\ndef sha_of(p):\n    hh = hashlib.sha256()\n    with open(p, \'rb\') as f:\n        for b in iter(lambda: f.read(8 << 20), b\'\'): hh.update(b)\n    return hh.hexdigest()\ndef get_file(z, n):\n    p = os.path.join(DATA, n)\n    if not os.path.exists(p): z.extract(n, DATA)\n    rel = n.split(\'can-train-and-test/\')[1]\n    assert (os.path.getsize(p), sha_of(p)) == H[rel], \'hash mismatch \' + rel\n    return p, rel\n\nNF, NN, NG = len(feats.FRAME_NAMES), len(feats.NODE_NAMES), len(feats.GLOBAL_NAMES)\ndef make(name, h, drop):\n    if name in (\'GraphSAGE\', \'GraphSAGE_rewired\'): return models.GraphSAGE(NN, h, NG, drop)\n    if name == \'DeepSets\': return models.DeepSets(NN, h, NG, drop)\n    if name == \'GRU\': return models.GRUNet(NF, h, NG, drop)\n\ndef windows(p, stride):\n    d = feats.windows_for_file(p, stride)\n    if d is None: return None\n    if MODE == \'d4\':\n        fr = d[\'frame\'].copy(); fr[..., D4_FRAME] = 0; d[\'frame\'] = fr\n        nd = d[\'node\'].copy(); nd[..., D4_NODE] = 0; d[\'node\'] = nd\n    d[\'struct\'] = feats.structural_features(d[\'src\'], d[\'dst\'], d[\'nmask\'])\n    return d\n\ndef flat_features(D, with_struct):   # identical to exp_tune / exp_final / exp_eval\n    m = D[\'nmask\'][..., None]; x = D[\'node\'].astype(np.float32); cnt = m.sum(1).clip(1)\n    mean = (x * m).sum(1) / cnt; std = np.sqrt(((x - mean[:, None]) ** 2 * m).sum(1) / cnt)\n    mn = np.where(m, x, np.inf).min(1); mx = np.where(m, x, -np.inf).max(1)\n    X = [mean, std, mn, mx, D[\'glob\']]\n    if with_struct: X.append(D[\'struct\'])\n    return np.concatenate(X, 1).astype(np.float32)\n\ndef norm_stats(D):\n    f = D[\'frame\'].astype(np.float32).reshape(-1, NF); msk = D[\'nmask\'].reshape(-1)\n    n = D[\'node\'].astype(np.float32).reshape(-1, NN)[msk]\n    return {\'fm\': f.mean(0), \'fs\': f.std(0) + 1e-6, \'nm\': n.mean(0), \'ns\': n.std(0) + 1e-6, \'gm\': D[\'glob\'].mean(0), \'gs\': D[\'glob\'].std(0) + 1e-6}\n\ndef to_gpu(D, N, with_y):\n    """float32; features constant in training set to 0 (as k05 v2). Returns tensors + input diagnostics."""\n    T = lambda a: torch.tensor(a, dtype=torch.float32, device=DEV)\n    fm, fs, nm_, ns, gm, gs = T(N[\'fm\']), T(N[\'fs\']), T(N[\'nm\']), T(N[\'ns\']), T(N[\'gm\']), T(N[\'gs\'])\n    kf, kn, kg = (fs > CONST_STD).float(), (ns > CONST_STD).float(), (gs > CONST_STD).float()\n    msk = torch.from_numpy(D[\'nmask\']).to(DEV)\n    out = {\'frame\': ((torch.from_numpy(D[\'frame\'].astype(np.float32)).to(DEV) - fm) / fs) * kf,\n           \'node\': (((torch.from_numpy(D[\'node\'].astype(np.float32)).to(DEV) - nm_) / ns) * msk.unsqueeze(-1)) * kn,\n           \'glob\': ((torch.from_numpy(D[\'glob\']).to(DEV) - gm) / gs) * kg, \'nmask\': msk,\n           \'src\': torch.from_numpy(D[\'src\']).to(DEV), \'dst\': torch.from_numpy(D[\'dst\']).to(DEV), \'n\': len(D[\'y\'])}\n    if with_y: out[\'y\'] = torch.from_numpy(D[\'y\'].astype(np.float32)).to(DEV)\n    return out\n\ndef batch(T, ix, rewire, gen):\n    b = {\'frame\': T[\'frame\'][ix], \'node\': T[\'node\'][ix], \'nmask\': T[\'nmask\'][ix], \'glob\': T[\'glob\'][ix]}\n    dst = T[\'dst\'][ix]\n    if rewire: dst = models.rewire_dst(dst, gen)\n    b[\'adj\'] = models.build_adj(T[\'src\'][ix], dst, len(ix), DEV)\n    return b\n\ndef score_neural(m, T, rewire):\n    m.eval(); g = torch.Generator(device=DEV); g.manual_seed(EVAL_REWIRE_SEED); out = []\n    with torch.no_grad():\n        for s in range(0, T[\'n\'], 2048):\n            ix = torch.arange(s, min(s + 2048, T[\'n\']), device=DEV)\n            out.append(m(batch(T, ix, rewire, g)).float())\n    return torch.cat(out).cpu().numpy().astype(np.float32)\n\nKEYS = [\'frame\', \'node\', \'nmask\', \'src\', \'dst\', \'glob\', \'y\', \'struct\']\n\ndef run_set(st):\n    sd = os.path.join(OUT, st); os.makedirs(sd, exist_ok=True)\n    if os.path.exists(os.path.join(sd, \'DONE\')): log(st, \'already done\'); return\n    SEL = json.load(open(glob.glob(f\'/kaggle/input/**/tune/{st}/selection.json\', recursive=True)[0]))\n    t0 = time.time()\n    with zipfile.ZipFile(ZIP) as z:\n        allnames = z.namelist()\n        trn = sorted(n for n in allnames if n.startswith(f\'can-train-and-test/{st}/train_01/\') and n.endswith(\'.csv\'))\n        F = [windows(get_file(z, n)[0], TRAIN_STRIDE) for n in trn]\n    F = [d for d in F if d is not None]\n    D = {k: np.concatenate([d[k] for d in F]) for k in KEYS}; del F\n    log(st, \'train files\', len(trn), \'windows\', len(D[\'y\']), \'pos\', int(D[\'y\'].sum()), \'W\', W, \'stride\', TRAIN_STRIDE, \'s\', round(time.time() - t0, 1))\n    N = norm_stats(D)\n    const = {\'frame\': [feats.FRAME_NAMES[i] for i in np.flatnonzero(N[\'fs\'] <= CONST_STD)], \'node\': [feats.NODE_NAMES[i] for i in np.flatnonzero(N[\'ns\'] <= CONST_STD)],\n             \'glob\': [feats.GLOBAL_NAMES[i] for i in np.flatnonzero(N[\'gs\'] <= CONST_STD)]}\n    log(st, \'training-constant features (zeroed):\', const)\n    meta = {\'mode\': MODE, \'W\': W, \'train_stride\': TRAIN_STRIDE, \'eval_stride\': EVAL_STRIDE, \'train_windows\': int(len(D[\'y\'])), \'train_pos\': int(D[\'y\'].sum()),\n            \'constant_in_training\': const, \'models\': {}}\n    # ---- train (frozen k03 selections, no tuning) ----\n    import lightgbm as lgb\n    LGB = {}\n    for name, ws in ((\'LightGBM\', False), (\'LightGBM_S\', True)):\n        X = flat_features(D, ws); y = D[\'y\']; pos = y.sum(); neg = len(y) - pos; LGB[name] = []\n        for seed in SEEDS:\n            clf = lgb.LGBMClassifier(n_estimators=SEL[name][\'final_epochs\'], scale_pos_weight=neg / max(pos, 1), n_jobs=args.lgb_threads, verbose=-1,\n                                     random_state=seed, **SEL[name][\'config\'])\n            clf.fit(X, y); LGB[name].append(clf.booster_)\n            clf.booster_.save_model(os.path.join(sd, f\'{name}_seed{seed}.txt\'))\n        meta[\'models\'][name] = {\'config\': SEL[name][\'config\'], \'n_estimators\': SEL[name][\'final_epochs\']}\n        log(st, name, \'trained 5 seeds\'); del X\n    T = to_gpu(D, N, True); n = T[\'n\']; pos = float(T[\'y\'].sum()); neg = n - pos; NNET = {}\n    for name in (\'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\'):\n        sel = SEL[name]; cfg = sel[\'config\']; h = sel[\'hidden\']; E = sel[\'final_epochs\']; NNET[name] = []\n        for seed in SEEDS:\n            torch.manual_seed(seed); np.random.seed(seed)\n            gen = torch.Generator(device=DEV); gen.manual_seed(seed)\n            m = make(name, h, cfg[\'dropout\']).to(DEV); opt = torch.optim.AdamW(m.parameters(), lr=cfg[\'lr\'])\n            lossf = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1.0), device=DEV)); rew = name == \'GraphSAGE_rewired\'; t = time.time()\n            for ep in range(E):\n                m.train(); perm = torch.randperm(n, device=DEV, generator=gen); tot = 0.0\n                for s in range(0, n, BS):\n                    ix = perm[s:s + BS]\n                    loss = lossf(m(batch(T, ix, rew, gen)), T[\'y\'][ix]); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(ix)\n            assert np.isfinite(tot), f\'non-finite training loss {st} {name} {seed}\'\n            torch.save(m.state_dict(), os.path.join(sd, f\'{name}_seed{seed}.pt\')); m.eval(); NNET[name].append(m)\n            log(st, name, \'seed\', seed, \'epochs\', E, \'params\', models.nparams(m), \'last loss\', round(tot / n, 6), \'s\', round(time.time() - t, 1))\n        meta[\'models\'][name] = {\'config\': cfg, \'hidden\': h, \'epochs\': E, \'params\': models.nparams(NNET[name][0])}\n    del T; torch.cuda.empty_cache(); del D\n    json.dump(meta, open(os.path.join(sd, \'train_meta.json\'), \'w\'), indent=1, default=str)\n    # ---- evaluate ----\n    train_sha = {H[r][1] for r in H if r.startswith(f\'{st}/train_01/\')}\n    with zipfile.ZipFile(ZIP) as z:\n        allnames = z.namelist()\n        cells = sorted({n.split(\'/\')[2] for n in allnames if n.startswith(f\'can-train-and-test/{st}/test_\') and n.endswith(\'.csv\')})\n        for cell in cells:\n            short = cell[:7]\n            if (st, short) in EXCLUDED: continue\n            if MODE != \'d4\' and short != \'test_02\': continue\n            names = sorted(n for n in allnames if n.startswith(f\'can-train-and-test/{st}/{cell}/\') and n.endswith(\'.csv\'))\n            parts = {k: [] for k in KEYS + [\'nattack\', \'starts\']}; fid = []; finfo = []\n            for i, nm in enumerate(names):\n                p, rel = get_file(z, nm); assert H[rel][1] not in train_sha, \'LEAKAGE \' + rel\n                d = windows(p, EVAL_STRIDE)\n                for k in parts: parts[k].append(d[k])\n                fid.append(np.full(len(d[\'y\']), i, np.int16))\n                finfo.append({\'file\': os.path.basename(nm), \'relative_path\': rel, \'sha256\': H[rel][1], \'token\': fam(nm), \'windows\': int(len(d[\'y\'])),\n                              \'pos\': int(d[\'y\'].sum()), \'hours\': float(d[\'t1\'].max() - d[\'t0\'].min()) / 3600.0})\n            Dt = {k: np.concatenate(v) for k, v in parts.items()}; fid = np.concatenate(fid)\n            S = {}\n            Xb = flat_features(Dt, False); Xs = flat_features(Dt, True)\n            S[\'Rule\'] = (SEL[\'Rule\'][\'sign\'] * Xb[:, SEL[\'Rule\'][\'feature_index\']])[None].astype(np.float32)\n            S[\'LightGBM\'] = np.stack([b.predict(Xb) for b in LGB[\'LightGBM\']]).astype(np.float32)\n            S[\'LightGBM_S\'] = np.stack([b.predict(Xs) for b in LGB[\'LightGBM_S\']]).astype(np.float32)\n            Tt = to_gpu(Dt, N, False)\n            for name, lst in NNET.items(): S[name] = np.stack([score_neural(m, Tt, name == \'GraphSAGE_rewired\') for m in lst])\n            del Tt; torch.cuda.empty_cache()\n            for m in MODELS: assert np.isfinite(S[m]).all(), f\'non-finite scores {st} {cell} {m}\'\n            np.savez_compressed(os.path.join(sd, f\'{short}_scores.npz\'), y=Dt[\'y\'].astype(np.int8), file_id=fid, nattack=Dt[\'nattack\'], starts=Dt[\'starts\'],\n                                **{f\'score_{m}\': S[m] for m in MODELS})\n            json.dump({\'set\': st, \'cell\': cell, \'mode\': MODE, \'W\': W, \'files\': finfo, \'windows\': int(len(Dt[\'y\'])), \'pos\': int(Dt[\'y\'].sum()),\n                       \'prevalence\': float(Dt[\'y\'].mean()), \'hours\': float(sum(f[\'hours\'] for f in finfo))},\n                      open(os.path.join(sd, f\'{short}_meta.json\'), \'w\'), indent=1)\n            log(st, cell, \'windows\', len(Dt[\'y\']), \'pos\', int(Dt[\'y\'].sum()))\n    open(os.path.join(sd, \'DONE\'), \'w\').write(\'ok\'); log(st, \'DONE\')\n\nfor st in SETS:\n    try:\n        run_set(st)\n    except Exception as e:\n        log(st, \'FATAL\', repr(e), traceback.format_exc()[-2000:])\n', 'exp_audit.py': '"""Stage 5c recording-level audit of the FROZEN k05 outputs. No training, no re-scoring, no exclusion of data.\nPurpose: test whether B-cell conclusions are robust to recording-level imbalance (2 recordings per attack token; one can dominate positives).\nUsage: python exp_audit.py --eval <k05 eval dir> --out /kaggle/working/audit"""\nimport os, sys, json, glob, argparse, itertools, math\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\n\nMODELS = [\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\']\nCONTRASTS = {\'H1_GS_minus_DeepSets\': (\'GraphSAGE\', \'DeepSets\'), \'H1a_GS_minus_rewired\': (\'GraphSAGE\', \'GraphSAGE_rewired\'),\n             \'H2_GS_minus_GRU\': (\'GraphSAGE\', \'GRU\'), \'H3_GS_minus_LGBS\': (\'GraphSAGE\', \'LightGBM_S\'),\n             \'ladder_LGBS_minus_LGB\': (\'LightGBM_S\', \'LightGBM\')}\nMARGIN = 0.02; MC_SUMMARY = 200000; MC_SEED = 20260917\n\ndef ap_prep(s, y, fid, nfiles):\n    o = np.argsort(-s, kind=\'stable\'); ss = s[o]; ys = y[o].astype(np.float64); fs = fid[o]\n    ends = np.r_[np.flatnonzero(ss[1:] != ss[:-1]), len(ss) - 1]\n    CP = np.zeros((nfiles, len(ends))); CN = np.zeros((nfiles, len(ends)))\n    for f in range(nfiles):\n        m = fs == f\n        CP[f] = np.cumsum(m * ys)[ends]; CN[f] = np.cumsum(m * (1 - ys))[ends]\n    return CP, CN\n\ndef ap_w(CP, CN, W, chunk=64):\n    out = np.empty(len(W))\n    for c in range(0, len(W), chunk):\n        w = W[c:c + chunk]\n        tp = w @ CP; fp = w @ CN; P = tp[:, -1:]; den = tp + fp\n        prec = np.where(den > 0, tp / np.where(den > 0, den, 1), 0.0)\n        rec = np.where(P > 0, tp / np.where(P > 0, P, 1), 0.0)\n        ap = (np.diff(rec, axis=1, prepend=0.0) * prec).sum(1)\n        out[c:c + chunk] = np.where(P[:, 0] > 0, ap, np.nan)\n    return out\n\ndef exact_token_resamples(tokens):\n    """Every distinct token-stratified bootstrap outcome with its probability. For a token with k recordings, the\n    multiset of k draws with replacement; probability = multinomial weight / k**k."""\n    tokens = np.asarray(tokens); per = []\n    for t in sorted(set(tokens.tolist())):\n        idx = np.flatnonzero(tokens == t); k = len(idx); opts = {}\n        for draw in itertools.product(range(k), repeat=k):\n            cnt = tuple(sorted(np.bincount(draw, minlength=k).tolist(), reverse=False))\n            key = tuple(np.bincount(draw, minlength=k).tolist())\n            opts[key] = opts.get(key, 0) + 1\n        per.append([(idx, np.array(key, float), c / float(k ** k)) for key, c in opts.items()])\n    W = []; P = []\n    for combo in itertools.product(*per):\n        w = np.zeros(len(tokens)); p = 1.0\n        for idx, cnt, pr in combo:\n            w[idx] = cnt; p *= pr\n        W.append(w); P.append(p)\n    return np.array(W), np.array(P)\n\ndef wq(vals, probs, q):\n    o = np.argsort(vals); v = vals[o]; c = np.cumsum(probs[o]); c /= c[-1]\n    return float(v[np.searchsorted(c, q, side=\'left\').clip(0, len(v) - 1)])\n\ndef summarise(vals, probs):\n    m = np.isfinite(vals); vals, probs = vals[m], probs[m] / probs[m].sum()\n    lo95, hi95, lo90, hi90 = wq(vals, probs, .025), wq(vals, probs, .975), wq(vals, probs, .05), wq(vals, probs, .95)\n    if lo95 > 0: d = \'superior\'\n    elif hi95 < 0: d = \'inferior\'\n    elif lo90 >= -MARGIN and hi90 <= MARGIN: d = \'practically_equivalent\'\n    else: d = \'inconclusive\'\n    p = min(1.0, 2 * min(probs[vals <= 0].sum(), probs[vals >= 0].sum()))\n    return {\'mean_over_resamples\': float((vals * probs).sum()), \'ci95\': [lo95, hi95], \'ci90\': [lo90, hi90],\n            \'decision\': d, \'p_exact_two_sided\': float(p), \'n_distinct_resamples\': int(len(vals))}\n\ndef main(EVAL, OUT):\n    os.makedirs(OUT, exist_ok=True); R = {\'cells\': {}, \'summary\': {}, \'method\': {\n        \'per_recording_ap\': \'AP computed on that recording alone (diagnostic only; unstable at low positive counts)\',\n        \'leave_one_out\': \'cell AP recomputed with that recording removed (replaces the ill-defined per-recording contribution to a global ranking metric)\',\n        \'exact_bootstrap\': \'complete enumeration of the token-stratified recording bootstrap with exact probabilities (replaces 2,000 random draws; same estimator)\',\n        \'no_exclusions\': \'no recording is dropped from the frozen results; leave-one-out is diagnostic\'}}\n    per_cell_dist = {}\n    for path in sorted(glob.glob(os.path.join(EVAL, \'set_*\', \'test_02_scores.npz\'))):\n        st = path.split(os.sep)[-2]; key = f\'{st}/test_02\'\n        Z = np.load(path); meta = json.load(open(path.replace(\'_scores.npz\', \'_meta.json\')))\n        y = Z[\'y\'].astype(np.int8); fid = Z[\'file_id\'].astype(np.int64); files = meta[\'files\']; nf = len(files)\n        tokens = [f[\'token\'] for f in files]\n        S = {m: Z[f\'score_{m}\'].astype(np.float64) for m in MODELS}\n        cell = {\'files\': [], \'aggregate\': {}, \'leave_one_out\': {}, \'exact_bootstrap\': {}, \'seed_vs_recording_variability\': {}}\n        tot_pos = int(y.sum())\n        for i, f in enumerate(files):\n            ix = fid == i; rec = {\'file\': f[\'file\'], \'token\': f[\'token\'], \'windows\': int(ix.sum()), \'pos\': int(y[ix].sum()),\n                                  \'share_of_cell_positives\': round(float(y[ix].sum()) / max(tot_pos, 1), 4),\n                                  \'prevalence\': round(float(y[ix].mean()), 5), \'hours\': round(f[\'hours\'], 4), \'ap_mean\': {}, \'ap_sd\': {}}\n            for m in MODELS:\n                a = [average_precision_score(y[ix], s[ix]) if y[ix].sum() > 0 else float(\'nan\') for s in S[m]]\n                rec[\'ap_mean\'][m] = round(float(np.mean(a)), 4); rec[\'ap_sd\'][m] = round(float(np.std(a, ddof=1)) if len(a) > 1 else 0.0, 4)\n            rec[\'contrasts\'] = {h: round(rec[\'ap_mean\'][a] - rec[\'ap_mean\'][b], 4) for h, (a, b) in CONTRASTS.items()}\n            cell[\'files\'].append(rec)\n        agg = {m: float(np.mean([average_precision_score(y, s) for s in S[m]])) for m in MODELS}\n        cell[\'aggregate\'] = {\'ap_mean\': {m: round(agg[m], 4) for m in MODELS},\n                             \'contrasts\': {h: round(agg[a] - agg[b], 4) for h, (a, b) in CONTRASTS.items()},\n                             \'windows\': int(len(y)), \'pos\': tot_pos, \'prevalence\': round(float(y.mean()), 5)}\n        for i, f in enumerate(files):   # leave-one-recording-out\n            keep = fid != i\n            if y[keep].sum() == 0: continue\n            a2 = {m: float(np.mean([average_precision_score(y[keep], s[keep]) for s in S[m]])) for m in MODELS}\n            cell[\'leave_one_out\'][f[\'file\']] = {\'ap_mean\': {m: round(a2[m], 4) for m in MODELS},\n                                                \'contrasts\': {h: round(a2[a] - a2[b], 4) for h, (a, b) in CONTRASTS.items()},\n                                                \'delta_vs_full\': {h: round((a2[a] - a2[b]) - (agg[a] - agg[b]), 4) for h, (a, b) in CONTRASTS.items()}}\n        W, P = exact_token_resamples(tokens)\n        B = {}\n        for m in MODELS:\n            acc = np.zeros(len(W))\n            for s in S[m]:\n                CP, CN = ap_prep(s, y, fid, nf)\n                chk = ap_w(CP, CN, np.ones((1, nf)))[0]\n                assert abs(chk - average_precision_score(y, s)) < 1e-9\n                acc += ap_w(CP, CN, W)\n            B[m] = acc / len(S[m])\n        per_cell_dist[key] = {\'W_prob\': P, \'ap\': B}\n        for h, (a, b) in CONTRASTS.items():\n            cell[\'exact_bootstrap\'][h] = {\'point_full_sample\': round(agg[a] - agg[b], 4), **summarise(B[a] - B[b], P)}\n        for m in MODELS:\n            sd_seed = float(np.std([average_precision_score(y, s) for s in S[m]], ddof=1)) if len(S[m]) > 1 else 0.0\n            sd_rec = float(np.sqrt(((B[m] - (B[m] * P).sum()) ** 2 * P).sum()))\n            cell[\'seed_vs_recording_variability\'][m] = {\'sd_across_seeds\': round(sd_seed, 4), \'sd_across_recording_resamples\': round(sd_rec, 4)}\n        R[\'cells\'][key] = cell\n        print(key, \'exact resamples\', len(W), \'aggregate\', cell[\'aggregate\'][\'contrasts\'], flush=True)\n    keys = sorted(per_cell_dist)\n    rng = np.random.default_rng(MC_SEED)\n    draws = {k: rng.choice(len(per_cell_dist[k][\'W_prob\']), size=MC_SUMMARY, p=per_cell_dist[k][\'W_prob\'] / per_cell_dist[k][\'W_prob\'].sum()) for k in keys}\n    strata = {\'B_summary\': keys, \'B_same_manufacturer\': [k for k in keys if k.startswith(\'set_01\')], \'B_cross_manufacturer\': [k for k in keys if not k.startswith(\'set_01\')]}\n    unif = np.full(MC_SUMMARY, 1.0 / MC_SUMMARY)\n    for h, (a, b) in CONTRASTS.items():\n        for sname, ks in strata.items():\n            d = np.mean([per_cell_dist[k][\'ap\'][a][draws[k]] - per_cell_dist[k][\'ap\'][b][draws[k]] for k in ks], 0)\n            pt = float(np.mean([R[\'cells\'][k][\'aggregate\'][\'contrasts\'][h] for k in ks]))\n            R[\'summary\'].setdefault(h, {})[sname] = {\'point_full_sample\': round(pt, 4), \'cells\': ks,\n                                                     **summarise(d, unif), \'note\': f\'{MC_SUMMARY} draws from the EXACT per-cell resample distributions\'}\n    json.dump(R, open(os.path.join(OUT, \'audit.json\'), \'w\'), indent=1)\n    # readable tables\n    L = []\n    for k, c in R[\'cells\'].items():\n        L.append(f"\\n== {k}  windows {c[\'aggregate\'][\'windows\']}  pos {c[\'aggregate\'][\'pos\']}  prevalence {c[\'aggregate\'][\'prevalence\']}")\n        L.append(\'recording\\ttoken\\twin\\tpos\\tposshare\\t\' + \'\\t\'.join(MODELS) + \'\\tGS-DS\\tGS-rew\')\n        for f in c[\'files\']:\n            L.append(f"{f[\'file\']}\\t{f[\'token\']}\\t{f[\'windows\']}\\t{f[\'pos\']}\\t{f[\'share_of_cell_positives\']}\\t" + \'\\t\'.join(f"{f[\'ap_mean\'][m]:.3f}" for m in MODELS)\n                     + f"\\t{f[\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f}\\t{f[\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f}")\n        L.append(\'AGGREGATE\\t\\t\\t\\t\\t\' + \'\\t\'.join(f"{c[\'aggregate\'][\'ap_mean\'][m]:.3f}" for m in MODELS)\n                 + f"\\t{c[\'aggregate\'][\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f}\\t{c[\'aggregate\'][\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f}")\n        L.append(\'leave-one-out (cell AP without that recording) — GS-DS / GS-rewired, and change vs full cell:\')\n        for fn, v in c[\'leave_one_out\'].items():\n            L.append(f"  drop {fn}\\tGS-DS {v[\'contrasts\'][\'H1_GS_minus_DeepSets\']:+.3f} ({v[\'delta_vs_full\'][\'H1_GS_minus_DeepSets\']:+.3f})"\n                     f"\\tGS-rew {v[\'contrasts\'][\'H1a_GS_minus_rewired\']:+.3f} ({v[\'delta_vs_full\'][\'H1a_GS_minus_rewired\']:+.3f})"\n                     f"\\tGS-LGB+S {v[\'contrasts\'][\'H3_GS_minus_LGBS\']:+.3f} ({v[\'delta_vs_full\'][\'H3_GS_minus_LGBS\']:+.3f})")\n        L.append(\'exact token-stratified bootstrap:\')\n        for h, v in c[\'exact_bootstrap\'].items():\n            L.append(f"  {h}\\tpoint {v[\'point_full_sample\']:+.4f}\\tmean {v[\'mean_over_resamples\']:+.4f}\\tci95 [{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\tci90 [{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\tp {v[\'p_exact_two_sided\']:.4f}\\tN {v[\'n_distinct_resamples\']}")\n        L.append(\'variability (SD across seeds | SD across recording resamples):\')\n        L.append(\'  \' + \'  \'.join(f"{m} {v[\'sd_across_seeds\']:.3f}|{v[\'sd_across_recording_resamples\']:.3f}" for m, v in c[\'seed_vs_recording_variability\'].items()))\n    L.append(\'\\n== B summary / strata (exact per-cell distributions)\')\n    L.append(\'contrast\\tstratum\\tpoint\\tmean\\tci95\\tci90\\tdecision\\tp\')\n    for h, dd in R[\'summary\'].items():\n        for sname, v in dd.items():\n            L.append(f"{h}\\t{sname}\\t{v[\'point_full_sample\']:+.4f}\\t{v[\'mean_over_resamples\']:+.4f}\\t[{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}]\\t[{v[\'ci90\'][0]:+.4f},{v[\'ci90\'][1]:+.4f}]\\t{v[\'decision\']}\\t{v[\'p_exact_two_sided\']:.4f}")\n    open(os.path.join(OUT, \'audit.tsv\'), \'w\').write(\'\\n\'.join(L) + \'\\n\'); print(\'\\n\'.join(L))\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser(); ap.add_argument(\'--eval\', required=True); ap.add_argument(\'--out\', default=\'/kaggle/working/audit\'); a = ap.parse_args()\n    main(a.eval, a.out)\n', 'robust_stats.py': '"""Stage 7 statistics. For each robustness mode: per-cell PR-AUC (5 seeds), per-attack-token PR-AUC (>=50 positives), exact token-stratified\nrecording bootstrap on the B cells (k06 code, unchanged), and the change versus the FROZEN k05 values. \'span\' = negatives-in-attack-span\nsensitivity built from the frozen k05 scores (evaluation-only). Nothing here alters RESULTS_FROZEN_v1.0."""\nimport os, sys, json, glob, shutil, argparse, io, contextlib\nimport numpy as np\nfrom sklearn.metrics import average_precision_score\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport exp_audit\nMODELS = [\'Rule\', \'LightGBM\', \'LightGBM_S\', \'DeepSets\', \'GRU\', \'GraphSAGE\', \'GraphSAGE_rewired\']\nH = [\'H1_GS_minus_DeepSets\', \'H1a_GS_minus_rewired\', \'H2_GS_minus_GRU\', \'H3_GS_minus_LGBS\', \'ladder_LGBS_minus_LGB\']\n\ndef build_span(k05_eval, out):\n    """Remove negative windows lying between the first and last positive window of each recording (B cells only)."""\n    rep = {}\n    for p in sorted(glob.glob(os.path.join(k05_eval, \'set_*\', \'test_02_scores.npz\'))):\n        st = p.split(os.sep)[-2]; os.makedirs(os.path.join(out, st), exist_ok=True)\n        Z = dict(np.load(p)); y = Z[\'y\']; fid = Z[\'file_id\'].astype(int); starts = Z[\'starts\']; keep = np.ones(len(y), bool)\n        for f in np.unique(fid):\n            ix = np.flatnonzero(fid == f); pos = ix[y[ix] == 1]\n            if len(pos) == 0: continue\n            lo, hi = starts[pos].min(), starts[pos].max()\n            drop = ix[(y[ix] == 0) & (starts[ix] > lo) & (starts[ix] < hi)]; keep[drop] = False\n        np.savez_compressed(os.path.join(out, st, \'test_02_scores.npz\'), **{k: v[keep] if v.ndim == 1 else v[:, keep] for k, v in Z.items()})\n        shutil.copy(p.replace(\'_scores.npz\', \'_meta.json\'), os.path.join(out, st, \'test_02_meta.json\'))\n        rep[f\'{st}/test_02\'] = {\'windows_before\': int(len(y)), \'negatives_removed\': int((~keep).sum()), \'windows_after\': int(keep.sum()), \'positives\': int(y.sum())}\n    return rep\n\ndef cell_tables(d):\n    R = {}\n    for p in sorted(glob.glob(os.path.join(d, \'set_*\', \'test_0*_scores.npz\'))):\n        st = p.split(os.sep)[-2]; key = f\'{st}/{os.path.basename(p)[:7]}\'\n        Z = np.load(p); meta = json.load(open(p.replace(\'_scores.npz\', \'_meta.json\')))\n        y = Z[\'y\']; fid = Z[\'file_id\'].astype(int); toks = [f[\'token\'] for f in meta[\'files\']]\n        c = {\'windows\': int(len(y)), \'pos\': int(y.sum()), \'prevalence\': float(y.mean()), \'models\': {}, \'per_token\': {}}\n        for m in MODELS:\n            a = [average_precision_score(y, s) for s in Z[f\'score_{m}\']]\n            c[\'models\'][m] = {\'ap_mean\': float(np.mean(a)), \'ap_sd\': float(np.std(a, ddof=1)) if len(a) > 1 else 0.0}\n        for t in sorted(set(toks)):\n            ti = np.isin(fid, [i for i, tt in enumerate(toks) if tt == t])\n            if y[ti].sum() < 50: continue\n            c[\'per_token\'][t] = {\'pos\': int(y[ti].sum()), **{m: float(np.mean([average_precision_score(y[ti], s[ti]) for s in Z[f\'score_{m}\']])) for m in MODELS}}\n        R[key] = c\n    return R\n\ndef main(ROBUST, K05_EVAL, K05_STATS, AUDIT_K06, OUT):\n    os.makedirs(OUT, exist_ok=True)\n    frozen = json.load(open(os.path.join(K05_STATS, \'results.json\')))[\'per_cell\']\n    frozen_ex = json.load(open(os.path.join(AUDIT_K06, \'audit.json\')))\n    RES = {\'span_filter\': build_span(K05_EVAL, os.path.join(OUT, \'span_eval\'))}\n    dirs = {\'span\': os.path.join(OUT, \'span_eval\')}\n    for m in (\'d4\', \'w32\', \'w128\'):\n        if glob.glob(os.path.join(ROBUST, m, \'set_*\', \'test_02_scores.npz\')): dirs[m] = os.path.join(ROBUST, m)\n    L = []\n    for mode, d in dirs.items():\n        cells = cell_tables(d)\n        n_b = len(glob.glob(os.path.join(d, \'set_*\', \'test_02_scores.npz\')))\n        with contextlib.redirect_stdout(io.StringIO()):\n            exp_audit.main(d, os.path.join(OUT, f\'audit_{mode}\'))\n        A = json.load(open(os.path.join(OUT, f\'audit_{mode}\', \'audit.json\')))\n        RES[mode] = {\'n_B_cells\': n_b, \'cells\': cells, \'exact_B\': {h: A[\'summary\'][h] for h in H},\n                     \'exact_B_per_cell\': {k: {h: c[\'exact_bootstrap\'][h] for h in H} for k, c in A[\'cells\'].items()},\n                     \'delta_vs_frozen_ap\': {k: {m: c[\'models\'][m][\'ap_mean\'] - frozen[k][\'models\'][m][\'ap_mean\'] for m in MODELS} for k, c in cells.items() if k in frozen}}\n        L.append(f\'\\n==== MODE {mode}  (B cells: {n_b})\')\n        L.append(\'cell\\t\' + \'\\t\'.join(MODELS) + \'\\t| change vs frozen: \' + \' \'.join(MODELS))\n        for k, c in cells.items():\n            dv = RES[mode][\'delta_vs_frozen_ap\'].get(k, {})\n            L.append(k + \'\\t\' + \'\\t\'.join(f"{c[\'models\'][m][\'ap_mean\']:.4f}" for m in MODELS) + \'\\t| \' + \' \'.join(f"{dv.get(m, float(\'nan\')):+.4f}" for m in MODELS))\n        L.append(\'exact-bootstrap B contrasts (frozen value in brackets):\')\n        for h in H:\n            for sname in (\'B_summary\', \'B_same_manufacturer\', \'B_cross_manufacturer\'):\n                v = A[\'summary\'][h].get(sname)\n                if not v: continue\n                fz = frozen_ex[\'summary\'][h][sname]\n                L.append(f"  {h}\\t{sname}\\t{v[\'point_full_sample\']:+.4f} [{v[\'ci95\'][0]:+.4f},{v[\'ci95\'][1]:+.4f}] {v[\'decision\']} p={v[\'p_exact_two_sided\']:.4f}\\t(frozen {fz[\'point_full_sample\']:+.4f} {fz[\'decision\']})")\n        L.append(\'per-cell H1 / H1a: \' + \'; \'.join(f"{k}: {v[\'H1_GS_minus_DeepSets\'][\'point_full_sample\']:+.4f} {v[\'H1_GS_minus_DeepSets\'][\'decision\']} / {v[\'H1a_GS_minus_rewired\'][\'point_full_sample\']:+.4f} {v[\'H1a_GS_minus_rewired\'][\'decision\']}"\n                                           for k, v in RES[mode][\'exact_B_per_cell\'].items()))\n        L.append(\'per-token PR-AUC on B cells (>=50 positives):\')\n        for k, c in cells.items():\n            if not k.endswith(\'test_02\'): continue\n            for t, v in c[\'per_token\'].items():\n                L.append(f"  {k}\\t{t}\\t{v[\'pos\']}\\t" + \' \'.join(f"{v[m]:.3f}" for m in MODELS))\n    L.insert(0, \'span filter: \' + json.dumps(RES[\'span_filter\']))\n    json.dump(RES, open(os.path.join(OUT, \'robust_results.json\'), \'w\'), indent=1)\n    open(os.path.join(OUT, \'robust_summary.tsv\'), \'w\').write(\'\\n\'.join(L) + \'\\n\'); print(\'\\n\'.join(L))\n\nif __name__ == \'__main__\':\n    ap = argparse.ArgumentParser()\n    for a in (\'robust\', \'k05_eval\', \'k05_stats\', \'k06_audit\', \'out\'): ap.add_argument(\'--\' + a, required=True)\n    a = ap.parse_args(); main(a.robust, a.k05_eval, a.k05_stats, a.k06_audit, a.out)\n'}
for n, s in FILES.items(): open('/kaggle/working/code/' + n, 'w').write(s)
g = lambda p: sorted(glob.glob(p, recursive=True))
Z = g('/kaggle/input/**/can-train-and-test-v1.zip'); HS = g('/kaggle/input/**/file_hashes_sha256.csv'); SEL = g('/kaggle/input/**/tune/set_0*/selection.json')
EV = g('/kaggle/input/**/eval/set_0*/test_02_scores.npz'); ST = g('/kaggle/input/**/stats/results.json'); AU = g('/kaggle/input/**/audit/audit.json')
print(Z, HS, SEL, EV, ST, AU, sep='\n')
assert len(Z) == 1 and len(HS) == 1 and len(SEL) == 4 and len(EV) == 4 and len(ST) == 1 and len(AU) == 1
K05_EVAL = os.path.dirname(os.path.dirname(EV[0])); K05_STATS = os.path.dirname(ST[0]); K06_AUDIT = os.path.dirname(AU[0])


In [ ]:
# smoke test of the W-generalised models on GPU before the long run
import sys, torch; sys.path.insert(0, '/kaggle/working/code')
import modelsW
for Wt in (32, 64, 128):
    modelsW.set_W(Wt); B = 4
    b = {'frame': torch.randn(B, Wt, 8, device='cuda:0'), 'node': torch.randn(B, Wt, 13, device='cuda:0'), 'nmask': torch.ones(B, Wt, dtype=torch.bool, device='cuda:0'),
         'glob': torch.randn(B, 4, device='cuda:0')}
    src = torch.randint(0, Wt, (B, Wt - 1), device='cuda:0', dtype=torch.uint8); dst = torch.randint(0, Wt, (B, Wt - 1), device='cuda:0', dtype=torch.uint8)
    b['adj'] = modelsW.build_adj(src, dst, B, 'cuda:0'); assert abs(float(b['adj'].sum(dim=(1, 2)).mean()) - 1.0) < 1e-5
    for m in (modelsW.DeepSets(13, 32, 4), modelsW.GraphSAGE(13, 32, 4), modelsW.GRUNet(8, 32, 4)):
        out = m.to('cuda:0')(b); assert out.shape == (B,) and torch.isfinite(out).all()
    print('W', Wt, 'smoke OK; edge weights per window sum to 1')


In [ ]:
import subprocess, sys, time
def chain(sets, dev):
    cmds = ' ; '.join(f'{sys.executable} /kaggle/working/code/exp_robust.py --mode {m} --sets {sets} --device {dev}' for m in ('d4', 'w32', 'w128'))
    return subprocess.Popen(['bash', '-c', cmds])
t0 = time.time(); pA = chain('set_01,set_03', 'cuda:0'); pB = chain('set_02,set_04', 'cuda:1')
print('exit codes', pA.wait(), pB.wait(), 'hours', round((time.time() - t0) / 3600, 2))
done = sorted(glob.glob('/kaggle/working/robust/*/set_0*/DONE')); print(len(done), 'of 12 set x mode runs DONE'); print(done)
logs = ''.join(open(p).read() for p in glob.glob('/kaggle/working/robust/*/log_*.txt'))
print('FATAL/ERROR lines:', [l[:300] for l in logs.split('\n') if 'FATAL' in l or 'ERROR' in l])


In [ ]:
t0 = time.time()
r = subprocess.run([sys.executable, '/kaggle/working/code/robust_stats.py', '--robust', '/kaggle/working/robust', '--k05_eval', K05_EVAL, '--k05_stats', K05_STATS,
                    '--k06_audit', K06_AUDIT, '--out', '/kaggle/working/robust_stats'])
print('stats exit', r.returncode, 'min', round((time.time() - t0) / 60, 1))
